###RAG PIPELINE - Data Ingestion to Vector DB pipeline

In [1]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


C:\Users\Mosra Pragna\AppData\Local\Temp\ipykernel_12016\775537409.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
c:\Users\Mosra Pragna\Desktop\Generative_AI_Learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#thisprogram gol is 
# Find every PDF inside a folder → load every PDF page as LangChain Documents → add source information → combine everything into one list.'''
### Read all the pdf's inside the directory
from pathlib import Path


def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf")) #program searches for .pdf files and converts the results into a Python list.
    
    print(f"Found {len(pdf_files)} PDF files to process") #prints total no.of pdfs
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            #PDF → LangChain Documents
            loader = PyPDFLoader(str(pdf_file))#str(pdf_file) converts the Path object into a string path. At this line, the PDF hasn't been loaded yet.#You've only created the loader.
            #Read each PDF and create LangChain Document objects.
            documents = loader.load()#THIS line actually loads the PDF
            
            # Add source information to metadata
            for doc in documents: #this loop handles each Document by Document/page by page
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            #Put Documents from the current PDF into the master list containing Documents from all PDFs.
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf_files")

Found 2 PDF files to process

Processing: AI_Generative.pdf
  ✓ Loaded 24 pages

Processing: ragquestions.pdf
  ✓ Loaded 14 pages

Total documents loaded: 38


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-03-01T18:28:01-05:00', 'moddate': '2025-03-01T18:28:01-05:00', 'source': '..\\data\\pdf_files\\AI_Generative.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1', 'source_file': 'AI_Generative.pdf', 'file_type': 'pdf'}, page_content='1 \n \nGenerative Artificial Intelligence for Academic Research : Evidence from  Guidance Issued for \nResearchers by Higher Education Institutions in the United States    \n \nAmrita Ganguly, Aditya Johri1, Areej Ali, and Nora McDonald \nGeorge Mason University \n \nAbstract \nThe recent development and use of generative AI (GenAI) has signaled a significant shift in research \nactivities such as brainstorming, proposal writing, dissemination, and even reviewing, resulting in questions \nabout how to balance the seemingly productive uses of GenAI with ethical concerns such as authorship and \ncopyright issues, use bia

In [5]:
print(all_pdf_documents[0].page_content)

1 
 
Generative Artificial Intelligence for Academic Research : Evidence from  Guidance Issued for 
Researchers by Higher Education Institutions in the United States    
 
Amrita Ganguly, Aditya Johri1, Areej Ali, and Nora McDonald 
George Mason University 
 
Abstract 
The recent development and use of generative AI (GenAI) has signaled a significant shift in research 
activities such as brainstorming, proposal writing, dissemination, and even reviewing, resulting in questions 
about how to balance the seemingly productive uses of GenAI with ethical concerns such as authorship and 
copyright issues, use biased training data, lack of transparency, and impact on user privacy. To address 
these concerns, many Higher Education Institutions ( HEIs) have released institutional gui dance for 
researchers. To better understand the guidance that is being provided we report findings from a thematic 
analysis of guidelines from thirty HEIs in the United States that are classified as R1 or “very h

In [6]:
len(all_pdf_documents)

38

In [7]:
print(all_pdf_documents[1].metadata)

{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-03-01T18:28:01-05:00', 'moddate': '2025-03-01T18:28:01-05:00', 'source': '..\\data\\pdf_files\\AI_Generative.pdf', 'total_pages': 24, 'page': 1, 'page_label': '2', 'source_file': 'AI_Generative.pdf', 'file_type': 'pdf'}


In [8]:
print(all_pdf_documents[1].page_content)

2 
 
3,838 postdocs indicated a similar level of engagement with GenAI, particularly chatbots, with 31% of 
respondents reporting using chatbots (Nordling, 2023). One application of GenAI in particular, Large 
Language Models (LLMs) based applications such as ChatGPT, has seen very high uptake as they can assist 
with writing which is a component of different parts of the research process (Biswas, 2023; Formosa et al., 
2024). Writing was already a task that was often undertaken with the help of tools such as Gramma rly, 
Zotero, and Evernote, among others that helped improve grammar and sentence structure and assisted with 
citations (Alkhaqani, 2023). The use of LLMs has now allowed researchers to use a single application for 
multiple writing related tasks in conjunction with functions such as data exploration and analysis 
(Abdelhafiz et al., 2024).  
 
Although the use of GenAI for research is on the rise, the advantages of the technology are unclear and 
there is ambiguity about 

In [9]:
#Document → Chunks
### Text splitting get into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    #Use the text splitter to split the documents, and store the resulting smaller documents in split_docs.
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [10]:
chunks=split_documents(all_pdf_documents)


Split 38 documents into 150 chunks

Example chunk:
Content: 1 
 
Generative Artificial Intelligence for Academic Research : Evidence from  Guidance Issued for 
Researchers by Higher Education Institutions in the United States    
 
Amrita Ganguly, Aditya Johri...
Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-03-01T18:28:01-05:00', 'moddate': '2025-03-01T18:28:01-05:00', 'source': '..\\data\\pdf_files\\AI_Generative.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1', 'source_file': 'AI_Generative.pdf', 'file_type': 'pdf'}


In [11]:
chunks

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-03-01T18:28:01-05:00', 'moddate': '2025-03-01T18:28:01-05:00', 'source': '..\\data\\pdf_files\\AI_Generative.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1', 'source_file': 'AI_Generative.pdf', 'file_type': 'pdf'}, page_content='1 \n \nGenerative Artificial Intelligence for Academic Research : Evidence from  Guidance Issued for \nResearchers by Higher Education Institutions in the United States    \n \nAmrita Ganguly, Aditya Johri1, Areej Ali, and Nora McDonald \nGeorge Mason University \n \nAbstract \nThe recent development and use of generative AI (GenAI) has signaled a significant shift in research \nactivities such as brainstorming, proposal writing, dissemination, and even reviewing, resulting in questions \nabout how to balance the seemingly productive uses of GenAI with ethical concerns such as authorship and \ncopyright issues, use bia

###Embedding and vector database

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
#chunks to enbedding
#Embedding model = converts text into numbers that capture useful semantic relationships.
#ChromaDB needs a numerical representation of the text for semantic search.
#to put related operations like 1. Load embedding model , 2. Generate embeddings,3. Keep model information here, we create a class called EmbeddingManager that encapsulates these functionalities.
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"): #all-MiniLM-L6-v2, all-mpnet-base-v2,BAAI/bge-small-en-v1.5, BAAI/bge-base-en-v1.5 are some of the other embeding models that can be used. The choice of model affects the quality and dimensionality of the embeddings.
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name) #the model is loaded into memory and ready to generate embeddings for text data.
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")#asking like how many numbers you produce for each text input. This is important for downstream tasks like similarity search or clustering.
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings




In [14]:
## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7142.72it/s]


Model loaded successfully. Embedding dimension: 384


###VECTOR STORE

In [15]:

#VectorStore = a place where we store vectors (embeddings).
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)

            #his is where we connecting our Python code to ChromaDB.
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG","hnsw:space": "cosine"}#we are specifying that we want to use cosine similarity as the distance metric for our vector search. This is important because it affects how the vector store will compare embeddings when performing searches.than default euclidean distance. By using cosine similarity, we are focusing on the orientation of the vectors rather than their magnitude, which is often more meaningful for text embeddings.
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}") #tells  how many records currently exist.
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise 


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection ,here we are adding the documents, their embeddings, and metadata to the ChromaDB collection. The add method of the collection takes care of storing this information in the vector store.
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise




#flow of this program is as follows:
'''Create a VectorStore
        ↓
Create a folder for ChromaDB
        ↓
Connect to persistent ChromaDB
        ↓
Get/create "pdf_documents" collection
        ↓
Wait for documents + embeddings
        ↓
For every document:
    create unique ID
    collect metadata
    collect text
    collect embedding
        ↓
Send everything to ChromaDB
        ↓
Now the PDF chunks are stored
and can later be searched'''


'Create a VectorStore\n        ↓\nCreate a folder for ChromaDB\n        ↓\nConnect to persistent ChromaDB\n        ↓\nGet/create "pdf_documents" collection\n        ↓\nWait for documents + embeddings\n        ↓\nFor every document:\n    create unique ID\n    collect metadata\n    collect text\n    collect embedding\n        ↓\nSend everything to ChromaDB\n        ↓\nNow the PDF chunks are stored\nand can later be searched'

In [16]:
vectorstore=VectorStore()


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [17]:
vectorstore

In [18]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]#Take the text out of every document chunk.


In [19]:

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)



Generating embeddings for 150 texts...


Batches: 100%|██████████| 5/5 [00:05<00:00,  1.18s/it]

Generated embeddings with shape: (150, 384)


In [20]:
##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Adding 150 documents to vector store...
Successfully added 150 documents to vector store
Total documents in collection: 150


now chroma db contains as follows:
┌─────────────────────────────────────────────┐
│              pdf_documents                  │
├──────────┬──────────────┬───────────────────┤
│ ID       │ Text         │ Embedding         │
├──────────┼──────────────┼───────────────────┤
│ doc_001  │ RAG is...    │ [0.12, 0.45,...]  │
│ doc_002  │ Embeddings.. │ [0.67, 0.21,...]  │
│ doc_003  │ Retrieval... │ [0.31, 0.82,...]  │
└──────────┴──────────────┴───────────────────┘

Retriever Pipeline From VectorStore

In [21]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager


    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]: #top_k means "Give me the 5 most similar chunks.
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]


    # Search in vector store
        try:
            #"ChromaDB, take this query vector and find the most similar vectors that you have stored.
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],#The outer list is because Chroma's query API accepts multiple query embeddings.
                n_results=top_k
            )
            print("Distances returned by ChromaDB:")
            print(results["distances"])
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0] #these are the actual text chunks that were retrieved based on the query.
                metadatas = results['metadatas'][0] #IT TELLS US THE METADATA LIKE PAGE NUMBER , ETC OF THE DOCUMENTS THAT WERE RETRIEVED.
                distances = results['distances'][0] #these are the distances between the query embedding and the embeddings of the retrieved documents. A smaller distance indicates a higher similarity.
                ids = results['ids'][0] #these are document IDs that were generated when the documents were added to the vector store.
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                                'id': doc_id,
                                'content': document,
                                'metadata': metadata,
                                'similarity_score': similarity_score,
                                'distance': distance,
                                'rank': i + 1
                            })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []



#THE FLOW OF THE RAG RETRIEVAL PROCESS IS AS FOLLOWS:
'''
User asks:
"What is RAG?"
       ↓
Generate embedding for question
       ↓
Query vector
       ↓
Send query vector to ChromaDB
       ↓
ChromaDB compares it with
stored document vectors
       ↓
Calculate distances
       ↓
Find closest vectors
       ↓
Return Top-K results
       ↓
Get their:
  • IDs
  • text
  • metadata
  • distances
       ↓
Convert distance → similarity score
       ↓
Apply threshold
       ↓
Return relevant document chunks'''

'\nUser asks:\n"What is RAG?"\n       ↓\nGenerate embedding for question\n       ↓\nQuery vector\n       ↓\nSend query vector to ChromaDB\n       ↓\nChromaDB compares it with\nstored document vectors\n       ↓\nCalculate distances\n       ↓\nFind closest vectors\n       ↓\nReturn Top-K results\n       ↓\nGet their:\n  • IDs\n  • text\n  • metadata\n  • distances\n       ↓\nConvert distance → similarity score\n       ↓\nApply threshold\n       ↓\nReturn relevant document chunks'

In [23]:
rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [24]:
rag_retriever.retrieve("how were DBSCA hyperparameter values selected?")

Retrieving documents for query: 'how were DBSCA hyperparameter values selected?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.35it/s]

Generated embeddings with shape: (1, 384)
Distances returned by ChromaDB:
[[0.5467546582221985, 0.6063345670700073, 0.6689214706420898, 0.6927350163459778, 0.696006715297699]]
Retrieved 5 documents (after filtering)


[{'id': 'doc_cc22c3fd_149',
  'content': 'evals_per_epoch 5 Number of evaluations to perform per epoch.\nsaves_per_epoch 3 Number of times to save checkpoints per epoch.\nTable 5: All training hyperparameters.\n14',
  'metadata': {'content_length': 168,
   'source_file': 'ragquestions.pdf',
   'trapped': '/False',
   'creationdate': '',
   'source': '..\\data\\pdf_files\\ragquestions.pdf',
   'doi': 'https://doi.org/10.48550/arXiv.2506.18027',
   'license': 'http://creativecommons.org/licenses/by/4.0/',
   'file_type': 'pdf',
   'title': 'PDF Retrieval Augmented Question Answering',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1',
   'total_pages': 14,
   'page': 13,
   'author': 'Thi Thu Uyen Hoang; Meenakshi Rajendran; Kun Zhang; Yuhan Wu; Viet Anh Nguyen',
   'creator': 'arXiv GenPDF (tex2pdf:a6404ea)',
   'doc_index': 149,
   'producer': 'pikepdf 8.15.1',
   'page_label': '14',
   'arxivid': 'https://arxiv.org/abs/2506.

In [25]:
rag_retriever.retrieve("what is the abstract of Generative Artificial Intelligence for Academic Research")

Retrieving documents for query: 'what is the abstract of Generative Artificial Intelligence for Academic Research'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 48.67it/s]

Generated embeddings with shape: (1, 384)
Distances returned by ChromaDB:
[[0.25607752799987793, 0.3131815791130066, 0.328596830368042, 0.3413565158843994, 0.34759896993637085]]
Retrieved 5 documents (after filtering)


[{'id': 'doc_24bb79db_0',
  'content': '1 \n \nGenerative Artificial Intelligence for Academic Research : Evidence from  Guidance Issued for \nResearchers by Higher Education Institutions in the United States    \n \nAmrita Ganguly, Aditya Johri1, Areej Ali, and Nora McDonald \nGeorge Mason University \n \nAbstract \nThe recent development and use of generative AI (GenAI) has signaled a significant shift in research \nactivities such as brainstorming, proposal writing, dissemination, and even reviewing, resulting in questions \nabout how to balance the seemingly productive uses of GenAI with ethical concerns such as authorship and \ncopyright issues, use biased training data, lack of transparency, and impact on user privacy. To address \nthese concerns, many Higher Education Institutions ( HEIs) have released institutional gui dance for \nresearchers. To better understand the guidance that is being provided we report findings from a thematic',
  'metadata': {'doc_index': 0,
   'content